# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [8]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [20]:
# Keep fossil sources only
df_fossil = df[df['Source_Type'] == 'Fossil']

# One colour per fossil source (course palette, CVD-safe)
fossil_colors = {
    'Coal':        '#2E75B6',   # blue
    'Oil':         '#E07B39',   # orange
    'Natural Gas': '#F5C842',   # yellow
}

# ── Step 1: Plotly Express base chart 
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    labels={'Source': 'Fossil source', 'TWh': 'Energy (TWh)'},
    height=700, width=1100,
    template='plotly_white'
)

# ── Step 2: Customisation 
fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh'
)

fig.update_layout(
    title='Asia uses more than twice as much fossil energy as any other region',
    font=dict(family='Arial', size=12)
)

# Grey out parent nodes (Region, Country) — keep only the three source colours
fig.data[0].marker.colors = [
    c if c in fossil_colors.values() else '#DDDDDD'
    for c in fig.data[0].marker.colors
]

fig.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [10]:
# Nuclear + Hydro, total TWh per country
low_carbon = (df[df['Source_Type'] == 'Low-carbon']
              .groupby('Country')['TWh'].sum()
              .reset_index()
              .sort_values('TWh', ascending=False))
print(low_carbon)

top_country = low_carbon.iloc[0]['Country']
top_twh = low_carbon.iloc[0]['TWh']

           Country     TWh
4           France  7593.0
9           Norway  6174.0
2           Canada  5969.0
1           Brazil  4975.0
11     South Korea  2484.0
6            India  1336.0
13   United States  1137.0
7            Japan  1012.0
12  United Kingdom   993.0
5          Germany   929.0
8           Mexico   906.0
3            China   882.0
10    South Africa   726.0
0        Australia   448.0


In [11]:
# Treemap
fig = px.treemap(
    low_carbon,
    path=[px.Constant('Low-carbon'), 'Country'],
    values='TWh',
    color_discrete_sequence=['#2E75B6'],
    labels={'TWh': 'Energy (TWh)'},
    height=600,
    template='plotly_white'
)

fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh'
)

fig.update_layout(
    title='Low-carbon energy (nuclear + hydro) by country',
    font=dict(family='Arial', size=12)
)

fig.show()

In [12]:
# Horizontal bar chart, sorted (largest at the top)
fig = px.bar(
    low_carbon.sort_values('TWh'),
    x='TWh', y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=['#2E75B6'],
    labels={'TWh': 'Energy (TWh)'},
    height=600,
    template='plotly_white'
)

fig.update_traces(texttemplate='%{x:,.0f} TWh', textposition='outside', cliponaxis=False)

fig.update_layout(
    title=f'{top_country} produces the most low-carbon energy: {top_twh:,.0f} TWh',
    font=dict(family='Arial', size=12),
    xaxis=dict(visible=False),
    yaxis=dict(title=''),
    margin=dict(r=100)
)

fig.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

***The bar chart is easier to interpret because all bars start from the same zero baseline and are sorted by value. This makes it simple to compare countries and immediately identify the leader, France, with 7,593 TWh. In contrast, the treemap requires comparing areas, which is more difficult when values are close. However, the treemap is still useful for getting a quick overview of each country's contribution to the total.***

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [15]:
# ── Data prep 
mix = df_fossil.groupby(['Region', 'Source'])['TWh'].sum().reset_index()
mix['Share'] = mix['TWh'] / mix.groupby('Region')['TWh'].transform('sum')   

# Sort regions by coal share, highest first (a region with no coal at all gets 0)
coal_share = (mix[mix['Source'] == 'Coal'].set_index('Region')['Share']
              .reindex(mix['Region'].unique()).fillna(0)
              .sort_values(ascending=False))
region_order = coal_share.index.tolist()
top_region = coal_share.index[0]
top_share = coal_share.iloc[0]

# Data labels only on the Coal segments
mix['Label'] = np.where(mix['Source'] == 'Coal', mix['Share'].map('{:.0%}'.format), '')

# One accent colour (Coal, same blue as Task 1); two greys for the rest
accent_mapping = {
    'Coal':        '#2E75B6',
    'Oil':         '#BBBBBB',
    'Natural Gas': '#DDDDDD',
}

# ── Step 1: Plotly Express base chart 
fig = px.bar(
    data_frame=mix,
    x='Share', y='Region',
    color='Source',
    text='Label',
    color_discrete_map=accent_mapping,
    category_orders={'Region': region_order,
                     'Source': ['Coal', 'Oil', 'Natural Gas']},   
    labels={'Share': 'Share of fossil energy', 'Source': 'Fossil source'},
    hover_name='Region',
    hover_data={'Label': False, 'Share': ':.2f'},
    height=600, width=1000,
    template='plotly_white'
)

# ── Step 2: Customisation 
fig.update_traces(
    textposition='inside', insidetextanchor='start',
    textfont=dict(color='white', size=12),
    marker_line_color='white', marker_line_width=1
)

fig.update_layout(
    title=f'{top_region} relies most on coal: {top_share:.0%} of its fossil energy',
    font=dict(family='Arial', size=12),
    bargap=0.25,
    legend=dict(orientation='h', x=0, y=1.08),
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(title='', categoryorder='array', categoryarray=region_order[::-1])   
)

# Mark the end of the scale: every bar adds up to 100%
fig.add_annotation(
    x=1, y=0, xref='x', yref='paper',
    xanchor='right', yanchor='top', yshift=-4,
    text='100%', showarrow=False,
    font=dict(size=11, color='grey')
)

fig.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

The Task 1 treemap answer: *Where is the most fossil energy being used, and which countries and energy sources contribute the most?* It shows the absolute energy consumption in TWh.

The 100% stacked bar chart answer: *Which region depends most on coal?* Since all bars have the same length, we can compare the energy mix rather than the total consumption. Africa is the most dependent on coal, with 78% of its fossil energy coming from coal, even though Asia consumes much more coal overall. The Middle East has no coal consumption in this dataset, making it the least coal-dependent region.